# Learn Text Embeddings in Python (Without a Vector Database)

This notebook demonstrates how to turn text into embedding vectors and retrieve the most semantically similar document using cosine similarity.

**What you'll learn**
1. Load a pretrained embedding model.
2. Embed a small collection of documents.
3. Embed a user query.
4. Calculate cosine similarity with NumPy.
5. Retrieve the best-matching document.
6. Experiment with different queries.

**No vector database is used.** The embeddings are kept in a normal Python variable in notebook memory.

## 1. Install the required libraries

- sentence-transformers 

- numpy

## 2. Import libraries and load the model

We'll use `all-MiniLM-L6-v2`, a compact sentence-embedding model that produces 384-dimensional vectors.

In [1]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")
print("Model loaded!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded!


## 3. Create a small collection of documents

These are the pieces of text we want to search.

In [2]:
documents = [
    "Python is a programming language",
    "Java is used for enterprise applications",
    "Python is popular for machine learning",
    "SQL is used to query databases"
]

for i, document in enumerate(documents):
    print(f"{i}: {document}")

0: Python is a programming language
1: Java is used for enterprise applications
2: Python is popular for machine learning
3: SQL is used to query databases


## 4. Convert documents into embeddings

`model.encode()` converts each sentence into a numerical vector. The vectors below are stored in a NumPy array in memory—not in a vector database.

In [3]:
embeddings = model.encode(documents)

print("Embeddings shape:", embeddings.shape)
print("Number of documents:", len(embeddings))
print("Dimensions per embedding:", embeddings.shape[1])

# Inspect the first embedding (it contains 384 numbers)
print("\nFirst embedding:")
print(embeddings[0])

Embeddings shape: (4, 384)
Number of documents: 4
Dimensions per embedding: 384

First embedding:
[-3.53708453e-02  3.81650217e-02 -4.12600897e-02  1.60687733e-02
 -3.83670777e-02 -1.34470567e-01  3.57370079e-02  4.22967486e-02
 -3.46467867e-02 -2.64320057e-02 -4.44902517e-02  2.44822390e-02
  9.47517529e-02  1.70948785e-02  5.69991432e-02 -5.47181591e-02
 -8.70437250e-02 -4.23831772e-03  1.99949183e-02 -1.14943378e-01
 -3.67418043e-02  5.90621158e-02 -3.32976580e-02  2.30448879e-02
  1.57004390e-02 -3.24605522e-03 -9.96121671e-03 -1.88114792e-02
  2.31218617e-02 -2.39675841e-03 -5.23728654e-02  9.51938778e-02
  5.60904071e-02  3.42302136e-02  1.89513229e-02  5.95997870e-02
 -5.56279765e-03 -9.74243283e-02 -5.59518635e-02  3.30425706e-03
 -6.09469302e-02  1.64042991e-02 -5.19629195e-02 -3.67092416e-02
 -5.45357093e-02  5.61590046e-02 -2.10945737e-02 -8.86587054e-03
 -1.90217439e-02 -1.66452024e-02 -1.09764211e-01 -8.97258520e-03
 -3.70126516e-02 -7.99777880e-02 -3.40329390e-03 -1.17237

## 5. Embed a user query

The query must be encoded with the same embedding model as the documents.

In [4]:
query = "Which language is commonly used for machine learning?"
query_embedding = model.encode(query)

print("Query:", query)
print("Query embedding shape:", query_embedding.shape)

Query: Which language is commonly used for machine learning?
Query embedding shape: (384,)


## 6. Calculate cosine similarity

Cosine similarity compares the direction of two vectors. A larger score means the vectors are more similar under this model; it is **not** a probability.

The function below implements cosine similarity using NumPy.

In [5]:
def cosine_similarity(a, b):
    denominator = np.linalg.norm(a) * np.linalg.norm(b)
    if denominator == 0:
        return 0.0
    return float(np.dot(a, b) / denominator)

## 7. Compare the query with every document

Run this cell to see the similarity score for each document. The exact scores depend on the model and text.

In [6]:
scores = []

for document, embedding in zip(documents, embeddings):
    score = cosine_similarity(query_embedding, embedding)
    scores.append(score)
    print(f"{score:.4f}  |  {document}")

0.4026  |  Python is a programming language
0.3077  |  Java is used for enterprise applications
0.6467  |  Python is popular for machine learning
0.2939  |  SQL is used to query databases


## 8. Retrieve the most similar document

`np.argmax()` returns the index of the largest score.

In [7]:
best_index = int(np.argmax(scores))
best_score = scores[best_index]

print("Most relevant document:", documents[best_index])
print(f"Cosine similarity: {best_score:.4f}")

Most relevant document: Python is popular for machine learning
Cosine similarity: 0.6467


## 9. Try your own query

Change `query` below and run the cell. It re-embeds the query and searches the same in-memory document embeddings.

In [8]:
query = "How do I retrieve information from a database?"

query_embedding = model.encode(query)
scores = [
    cosine_similarity(query_embedding, embedding)
    for embedding in embeddings
]

ranked_indices = np.argsort(scores)[::-1]

print("Query:", query)
print("\nResults ranked by similarity:")
for rank, index in enumerate(ranked_indices, start=1):
    print(f"{rank}. ({scores[index]:.4f}) {documents[index]}")

Query: How do I retrieve information from a database?

Results ranked by similarity:
1. (0.5209) SQL is used to query databases
2. (0.0998) Java is used for enterprise applications
3. (0.0799) Python is a programming language
4. (0.0542) Python is popular for machine learning


## 10. Key concepts

- **Embedding model:** transforms text into a vector of numbers.
- **Embedding:** the vector representing a text's learned semantic features.
- **Cosine similarity:** compares the direction of two vectors.
- **Semantic search:** ranks text by vector similarity rather than relying only on exact keyword matches.
- **No vector DB:** we stored vectors in a NumPy array. This is useful for learning and small examples.
- **Scaling up:** for large collections, a vector index or vector database can make nearest-neighbor retrieval more efficient and provide persistence/metadata features.

### Important note
Embedding similarity is a retrieval signal, not a guarantee that a result is factually correct. In a RAG system, retrieved text is usually passed to a language model as context, and the response should be grounded in that context.